## Image Segmentation
Part 1 of image segment analysis, segmenting images with Python. Code from opensourceoptions.com resources and tutorial. https://opensourceoptions.com/python-geographic-object-based-image-analysis-geobia/

In [ ]:
# Importing Packages
import numpy as np
import gdal
from skimage import exposure
from skimage.segmentation import quickshift
import time

In [ ]:
# Reading in "NAIP" image file with gdal and numpy.
naip_fn = 'C:/temp/naip/m_4211161_se_12_1_20160624.tif'

# Reading in data from each of the four NAIP image bands (red, green, blue, and near-infrared).
driverTiff = gdal.GetDriverByName('GTiff')
naip_ds = gdal.Open(naip_fn)
nbands = naip_ds.RasterCount
band_data = []
print('bands', naip_ds.RasterCount, 'rows', naip_ds.RasterYSize, 'columns',
      naip_ds.RasterXSize)
for i in range(1, nbands+1):
    band = naip_ds.GetRasterBand(i).ReadAsArray()
    band_data.append(band)
band_data = np.dstack(band_data)

In [ ]:
# Segmentation done using scikit-image.

# scale image values from 0.0 - 1.0
img = exposure.rescale_intensity(band_data)

# do segmentation multiple options with quickshift and slic
# figure out which option works best (see examples right below)
seg_start = time.time()
# segments = quickshift(img, convert2lab=False)
# segments = quickshift(img, ratio=0.8, convert2lab=False)
# segments = quickshift(img, ratio=0.99, max_dist=5, convert2lab=False)
# segments = slic(img, n_segments=100000, compactness=0.1)
# segments = slic(img, n_segments=500000, compactness=0.01)
segments = slic(img, n_segments=500000, compactness=0.1)
print('segments complete', time.time() - seg_start)

# save segments to raster
segments_fn = 'C:/temp/naip/segments.tif'
segments_ds = driverTiff.Create(segments_fn, naip_ds.RasterXSize, naip_ds.RasterYSize,
                                1, gdal.GDT_Float32)
segments_ds.SetGeoTransform(naip_ds.GetGeoTransform())
segments_ds.SetProjection(naip_ds.GetProjectionRef())
segments_ds.GetRasterBand(1).WriteArray(segments)
segments_ds = None

In [ ]:
# Describing each segment based on spectral properties: 
# spectral properties = variables that will classify each segment as a land cover type.

# Function that returns the min, max, mean, variance, skewness, and kurtois for each band in a given array of pixels.
def segment_features(segment_pixels):
    features = []
    npixels, nbands = segment_pixels.shape
    for b in range(nbands):
        stats = scipy.stats.describe(segment_pixels[:, b])
        band_stats = list(stats.minmax) + list(stats)[2:]
        if npixels == 1:
            # in this case the variance = nan, change it 0.0
            band_stats[3] = 0.0
        features += band_stats
    return features

In [ ]:
# Makes a list of segment ID numbers and the pixels for each are passed to segment_features.
# Statistics from segment_features are stored in objects list, and object_id stored in a separate list.
# May take several hours to run, depending on setup. 

segment_ids = np.unique(segments)
objects = []
object_ids = []
for id in segment_ids:
    segment_pixels = img[segments == id]
    object_features = segment_features(segment_pixels)
    objects.append(object_features)
    object_ids.append(id)